# **Final Project - Toxcitiy Classification using AWS Sagemaker**

The objective of this project is to detect and classify toxic civil comments with the focus on ensuring fairness across different demographic groups. These civil comments can originate from online article discussion platforms, such as the comment sections found in some news articles. Toxicity includes comments that are harmful, abusive, and hateful, while non-toxic comments are respectful, even if they refer to some sensitive identities.   

This is fundamentally a text classification problem in Natural Language Processing. The project is more specifically a binary classification task that identifies whether a comment is either toxic or non-toxic. The dataset has distributional shifts across various demographic groups, ensuring that the model can handle unseen data in the real world.

In [1]:
!pip install unidecode
!pip install nltk

  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
Using cached Unidecode-1.4.0-py3-none-any.whl (235 kB)


In [15]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
import datetime
import time
import unidecode
import logging
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import io, os, sys, json, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from time import gmtime, strftime
from tqdm import tqdm
from sagemaker.workflow.pipeline_context import PipelineSession
from sklearn.pipeline import Pipeline


warnings.filterwarnings("ignore", category=UserWarning)

## Data Preparation

The raw data is stored as parquet files. The dataset was then divided into two major components: input data for training the model and batch data for real-time inference. Amazon S3 is used as a primary storage for managing all datasets throughout the machine learning pipeline. The input dataset is uploaded to S3 and then split into train, validation, and test subsets. The batch dataset, intended for real-time inference, is also stored in an S3 bucket. 

In [16]:
sagemaker_session = sagemaker.session.Session()
session = boto3.session.Session()
region = session.region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
bucket = sagemaker_session.default_bucket()
model_package_group_name = f"ToxicityModelPackageGroup"

# initialize bucket
s3 = boto3.Session().client(service_name="s3", region_name=region)
print("Default bucket: {}".format(bucket))

Default bucket: sagemaker-us-east-1-046536196377


In [17]:
# Verify the bucket creation
from botocore.client import ClientError

response = None

try:
    response = s3.head_bucket(Bucket=bucket)
    print(response)
    setup_s3_bucket_passed = True
except ClientError as e:
    print("[ERROR] Cannot find bucket {} in {} due to {}.".format(bucket, response, e))

{'ResponseMetadata': {'RequestId': 'BY00NC5SMHJWH1RB', 'HostId': 'phw07a+WQBNASQEc0jV6g0K+6WaxyNgNvsYMrsyq3tI+xzGVxI2DdPH4Y0Dc75HZQiabnMRTn6Q=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'phw07a+WQBNASQEc0jV6g0K+6WaxyNgNvsYMrsyq3tI+xzGVxI2DdPH4Y0Dc75HZQiabnMRTn6Q=', 'x-amz-request-id': 'BY00NC5SMHJWH1RB', 'date': 'Sat, 18 Oct 2025 22:02:33 GMT', 'x-amz-bucket-region': 'us-east-1', 'x-amz-access-point-alias': 'false', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'BucketRegion': 'us-east-1', 'AccessPointAlias': False}


In [18]:
account_id = boto3.client("sts").get_caller_identity().get("Account")
sm = boto3.Session().client(service_name="sagemaker", region_name=region)

In [19]:
base_uri = f"s3://{bucket}/toxicity-CICD"
print(base_uri)

s3://sagemaker-us-east-1-046536196377/toxicity-CICD


In [20]:
# Data source - Data for Toxicity classification is stored as parquet files
s3_private_path_parquet = f"{base_uri}/parquet".format(bucket)
print(s3_private_path_parquet)

s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet


### Copy data from github to S3 bucket

In [21]:
!aws s3 cp --recursive /home/sagemaker-user/aai540_toxicity_classification/civil $s3_private_path_parquet/ --include "*"


upload: civil/.ipynb_checkpoints/preprocessing-checkpoint.py to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/.ipynb_checkpoints/preprocessing-checkpoint.py
upload: civil/preprocessing.py to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/preprocessing.py
upload: civil/results_with_text.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/results_with_text.csv
upload: civil/test_input.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/test_input.csv
upload: civil/batch_data.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/batch_data.csv
upload: civil/test_predictions.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/test_predictions.csv
upload: civil/test_input_2.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/test_input_2.csv
upload: civil/test.csv to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet/test.csv
upload: civil/test_feature.parquet to s3://sagemaker-us-ea

#### List the files

In [22]:
print(s3_private_path_parquet)

s3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet


In [23]:
!aws s3 ls $s3_private_path_parquet/

                           PRE .ipynb_checkpoints/
                           PRE 046536196377/
2025-10-18 22:02:38    1408338 batch_data.csv
2025-10-18 22:02:38   27194395 input_data.csv
2025-10-18 22:02:38       1113 preprocessing.py
2025-10-18 22:02:38        390 results_with_text.csv
2025-10-15 17:38:18   34099179 test-00000-of-00001.parquet
2025-10-18 22:02:38    1370199 test.csv
2025-10-18 22:02:38     731321 test_feature.csv
2025-10-18 22:02:38     491770 test_feature.parquet
2025-10-18 22:02:38      64085 test_input.csv
2025-10-18 22:02:38      90321 test_input_2.csv
2025-10-18 22:02:38       4667 test_predictions.csv
2025-10-15 17:38:18   68844404 train-00000-of-00001.parquet
2025-10-18 22:02:38   20647510 train.csv
2025-10-18 22:02:38   11068214 train_feature.csv
2025-10-18 22:02:38    7277518 train_feature.parquet
2025-10-18 22:02:38    1833853 val_feature.parquet
2025-10-15 17:38:18   11697541 validation-00000-of-00001.parquet
2025-10-18 22:02:38    5177098 validation.csv
2

In [24]:
s3_private_path_parquet

's3://sagemaker-us-east-1-046536196377/toxicity-CICD/parquet'

The dataset, CivilComments-WILDS, from Huggingface (https://huggingface.co/datasets/pietrolesci/civilcomments-wilds), consists of 445K records across different demographics. The target variable is toxicity, which is identified by 0 (non-toxic) and 1 (toxic). 


In [25]:
# Your region & private bucket
s3_client = boto3.client("s3", region_name=region)
input_path = "civil/input_data.csv"
batch_path = "civil/batch_data.csv"


# Split the s3 path into bucket and prefix
bucket_name, prefix = s3_private_path_parquet.replace("s3://", "").split("/", 1)
print(bucket_name)
print(prefix)
# Define the file keys (relative to prefix)
train_file_key = f"{prefix}/train-00000-of-00001.parquet"
val_file_key = f"{prefix}/validation-00000-of-00001.parquet"
test_file_key = f"{prefix}/test-00000-of-00001.parquet"


# Load the files
train_data_object = s3_client.get_object(Bucket=bucket_name, Key=train_file_key)
validation_data_object = s3_client.get_object(Bucket=bucket_name, Key=val_file_key)
test_data_object = s3_client.get_object(Bucket=bucket_name, Key=test_file_key)

# Convert to pandas DataFrames
data = pd.concat(
    [
        pd.read_parquet(io.BytesIO(train_data_object["Body"].read())),
        pd.read_parquet(io.BytesIO(validation_data_object["Body"].read())),
        pd.read_parquet(io.BytesIO(test_data_object["Body"].read()))
    ]
)

data = pd.concat([data[data['toxicity'] == t].sample(n=35000, random_state=42) for t in [0, 1]])
# Split per class separately, then recombine
input_parts, batch_parts = [], []



for t in [0, 1]:
    class_subset = data[data['toxicity'] == t]
    split_idx = int(len(class_subset) * 0.95)
    input_parts.append(class_subset.iloc[:split_idx])
    batch_parts.append(class_subset.iloc[split_idx:])

# Combine balanced splits
input_data_df = pd.concat(input_parts).sample(frac=1, random_state=42).reset_index(drop=True)
batch_data_df = pd.concat(batch_parts).sample(frac=1, random_state=42).reset_index(drop=True)


input_data_df.to_csv(input_path)
batch_data_df.to_csv(batch_path)

print("Input Data Shape:", input_data_df.shape)
print("Batch Data Shape:", batch_data_df.shape)

input_data_uri = sagemaker.s3.S3Uploader.upload(local_path=input_path, desired_s3_uri=base_uri)
batch_data_uri = sagemaker.s3.S3Uploader.upload(local_path=batch_path, desired_s3_uri=base_uri)

print(input_data_uri)
print(batch_data_uri)

sagemaker-us-east-1-046536196377
toxicity-CICD/parquet
Input Data Shape: (66500, 21)
Batch Data Shape: (3500, 21)
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/input_data.csv
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/batch_data.csv


In [14]:
# split into train and test
TRAIN_PATH = "civil/train.csv"
VAL_PATH = "civil/validation.csv"
TEST_PATH = "civil/test.csv"

train, test = train_test_split(
    input_data_df,
    test_size=0.05,
    stratify=input_data_df['toxicity'],
    random_state=42
)

train, val = train_test_split(
    train,
    test_size=0.2,
    stratify=train['toxicity'],
    random_state=42
)

train.to_csv(TRAIN_PATH)
val.to_csv(VAL_PATH)
test.to_csv(TEST_PATH)

print("Train shape: ", train.shape)
print("Validation shape: ", val.shape)
print("Test shape: ", test.shape)

train_data_uri = sagemaker.s3.S3Uploader.upload(local_path=TRAIN_PATH, desired_s3_uri=base_uri)
val_data_data_uri = sagemaker.s3.S3Uploader.upload(local_path=VAL_PATH, desired_s3_uri=base_uri)
test_data_data_uri = sagemaker.s3.S3Uploader.upload(local_path=TEST_PATH, desired_s3_uri=base_uri)

Train shape:  (50540, 21)
Validation shape:  (12635, 21)
Test shape:  (3325, 21)


In [27]:
input_data_df.head()

,uid,id,comment_text,toxicity,has_active_attrs,active_attrs,male,female,LGBTQ,christian,...,other_religions,black,white,identity_any,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,355072,7140934,"Double, double toil and trouble; \n Fire bu...",0,False,[],0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,128269,450996,The Washington Post is trying to scare us with...,1,True,[muslim],0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
2,344723,872023,"Fools, all. So SAD, as their *President would...",1,False,[],0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,277361,978153,"I'm not gay, just slutty, where's my parade",1,True,[LGBTQ],0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,236167,5105982,Largest army ? I hope they all know how to swi...,1,False,[],0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Example

"If the ears that don't listen are black, it's a capital crime."	1  
"A vote for Hillary is a vote for globalism and the clubhouse of Democrats and Republicans."	0

## Data Preprocessing

The input text is cleaned, tokenized, and lemmatized, and is fed into the CI/CD pipeline 

In [15]:
!mkdir -p civil

#### Set up Sagemaker Feature store

In [16]:
from sagemaker.session import Session

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)

featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

In [17]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
import os
print(os.environ.get("AWS_DEFAULT_REGION"))

us-east-1


#### Data Preprocessing Utility Methods

**Text processing**  
The comment text is processed by cleaning the text, such as removing HTML tags, special characters, punctuations, and stop words, tokenization, and lemmatizing. The preprocessing is further applied to test and validation datasets.  

In [19]:
TAG_RE = re.compile(r'<[^>]+>')

def re_tags(text_list): #define remove tag funtion
    return [TAG_RE.sub('', str(word)).lower() for word in text_list]

def re_accented_char(text_list):
    return [unidecode.unidecode(word.encode().decode('utf-8')) for word in text_list]

# Extended contractions
def ex_contractions(text_list):
    result=[]
    for word in text_list:
        # replace contracting withoutsignal
        word = word.replace("wont","won't")
        word = word.replace("cant","can't")
        word = word.replace("its","it's")
        word = word.replace("youre","you're")
        word = word.replace("hes","he's")
        word = word.replace("shes","she's")
        word = word.replace("its","it's")
        word = word.replace("weare","we're")
        word = word.replace("theyre","they're")

        # specific
        word = re.sub(r"won\'t", "will not", str(word))
        word = re.sub(r"can\'t", "can not", str(word))

        # general
        word = re.sub(r"n\'t", " not", str(word))
        word = re.sub(r"\'re", " are", str(word))
        word = re.sub(r"\'s", " is", str(word))
        word = re.sub(r"\'d", " would", str(word))
        word = re.sub(r"\'ll", " will", str(word))
        word = re.sub(r"\'t", " not", str(word))
        word = re.sub(r"\'ve", " have", str(word))
        word = re.sub(r"\'m", " am", str(word))
        result.append(word)
    return result

# Removing Special Characters
def re_special_chars(text_list):
    return [re.sub("[^a-zA-Z0-9]"," ",word) for word in text_list]

# Lemmatization

def lemmatize_text(text_list):
    wnl = WordNetLemmatizer()
    lemmatizer_sentence = []  
    tokenizer=nltk.tokenize.WhitespaceTokenizer()
    for word in tokenizer.tokenize(text_list):
        lemmatizer_sentence.append(wnl.lemmatize(word,'v'))
        lemmatizer_sentence.append(" ")
    
    return("".join(lemmatizer_sentence))

def stopwords_text(text_list):
    stop = stopwords.words('english')
    sentence_without = []
    tokenizer=nltk.tokenize.WhitespaceTokenizer()
    for word in tokenizer.tokenize(text_list):
        if word not in stop:
            sentence_without.append(word)
            sentence_without.append(" ")
            
    return("".join(sentence_without))

def re_whitespaces(text_list): 
    result=[]
    for word in text_list:
        word=(re.sub(r'\d','',str(word))) #remove numbers 
        word = (re.sub(r'\s+',' ', str(word))) #remove duplicates white spacces
        result.append(word)
    return result

# Class for regular expressions application
class ApplyRegex(BaseEstimator, TransformerMixin):
    
    def __init__(self, regex_transformers):
        self.regex_transformers = regex_transformers
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        # Applying all regex functions in the regex_transformers dictionary
        for regex_name, regex_function in self.regex_transformers.items():
            X = regex_function(X)
            
        return X

class StopWordsRemoval(BaseEstimator, TransformerMixin):
    
    def __init__(self, text_stopwords):
        self.text_stopwords = text_stopwords
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return [self.text_stopwords(comment) for comment in X]

class LemmatizeProcess(BaseEstimator, TransformerMixin):
    
    def __init__(self, Lemmatize):
        self.Lemmatizer = Lemmatize
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        return [self.Lemmatizer(comment) for comment in X]

def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")
    return data_frame


def appendEventTime(df):
    # append EventTime feature
    df[event_time_feature_name] = pd.Series(
        [current_time_sec] * len(df), index = df.index, dtype="float64"
    )
    return df


def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")
    return data_frame


def appendEventTime(df):
    # append EventTime feature
    df[event_time_feature_name] = pd.Series(
        [current_time_sec] * len(df), index=df.index, dtype="float64"
    )
    return df

def featureGroupExists(feature_group_name):
    print(feature_group_name)
    # SageMaker client (not session)
    sm_client = boto3.client("sagemaker", region_name=region)
    featureGroupExists = False
    try:
        response = sm_client.describe_feature_group(FeatureGroupName=feature_group_name)
        featureGroupExists = True
        print(f"✅ Feature group exists. Status: {response['FeatureGroupStatus']}")
    except sm_client.exceptions.ResourceNotFound:
        featureGroupExists=False
        print(f"❌ {feature_group_name} does not exist.")
    return featureGroupExists

def wait_for_feature_group_creation_complete(feature_group):
    record_identifier_feature_name = "id"
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

def create_feature_group(feature_group, fg_Exists):
    fg_Exists = featureGroupExists(feature_group.name)
    if not fg_Exists:
        feature_group.create(
            s3_uri=f"{base_uri}/parquet",
            record_identifier_name=record_identifier_feature_name,
            event_time_feature_name=event_time_feature_name,
            role_arn=role,
            enable_online_store=True,
        )
        wait_for_feature_group_creation_complete(feature_group)
    else:
        print(f"{feature_group.name} already exists")  

In [20]:
prefix

'toxicity-CICD/parquet'

In [21]:
# Defining regex transformers to be applied
train_df = train[['id','comment_text','toxicity']]
val_df = val[['id','comment_text','toxicity']]
test_df = test[['id','comment_text','toxicity']]

regex_transformers = {
    'remove_tags': re_tags,
    'remove_accents': re_accented_char,
    'decontracted': ex_contractions,
    're_sc': re_special_chars,
    'whitespaces': re_whitespaces
}

# Building a text prep pipeline
text_prep_pipeline = Pipeline([
    ('regex', ApplyRegex(regex_transformers)),
    ('stopwords', StopWordsRemoval(stopwords_text)),
    ('lemmatize', LemmatizeProcess(lemmatize_text)),
])

print("Executing Text processing pipeline...")
train_df["comment_text"] = text_prep_pipeline.fit_transform(train_df[train_df.columns[1:2]].values)
val_df["comment_text"] = text_prep_pipeline.transform(val_df[train_df.columns[1:2]].values)
test_df["comment_text"] = text_prep_pipeline.transform(test_df[train_df.columns[1:2]].values)
print("Completed...")

Executing Text processing pipeline...


/tmp/ipykernel_158/3817422339.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["comment_text"] = text_prep_pipeline.fit_transform(train_df[train_df.columns[1:2]].values)
/opt/conda/lib/python3.12/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/tmp/ipykernel_158/3817422339.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

Completed...


/tmp/ipykernel_158/3817422339.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["comment_text"] = text_prep_pipeline.transform(test_df[train_df.columns[1:2]].values)


In [22]:
train_df.head()

,id,comment_text,toxicity
35770,472168,agree back ride mountain bike friends never en...,0
7565,1066048,le pen guilty reveal truth islamic extremists ...,0
37949,7100281,irt thomas e stuart hypocritical double standa...,1
4882,5333036,awesome lol,0
51418,5540638,racial profile n nyou mean like epd arrest two...,1


In [23]:
current_time_sec = int(round(time.time()))
record_identifier_feature_name = "id"
event_time_feature_name = "EventTime"

# cast object dtype to string. 
# The SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
train_feature_data = cast_object_to_string(train_df)
val_feature_data = cast_object_to_string(val_df)
test_feature_data = cast_object_to_string(test_df)



/tmp/ipykernel_158/4167008236.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame[label] = data_frame[label].astype("str").astype("string")
/tmp/ipykernel_158/4167008236.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_frame[label] = data_frame[label].astype("str").astype("string")
/tmp/ipykernel_158/4167008236.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

In [24]:
# append event time
train_df = appendEventTime(train_feature_data)
val_df = appendEventTime(val_feature_data)
test_df = appendEventTime(test_feature_data)

/tmp/ipykernel_158/4167008236.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[event_time_feature_name] = pd.Series(
/tmp/ipykernel_158/4167008236.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[event_time_feature_name] = pd.Series(
/tmp/ipykernel_158/4167008236.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs

In [25]:
train_df.head()

,id,comment_text,toxicity,EventTime
35770,472168,agree back ride mountain bike friends never en...,0,1.760623e+09
7565,1066048,le pen guilty reveal truth islamic extremists ...,0,1.760623e+09
37949,7100281,irt thomas e stuart hypocritical double standa...,1,1.760623e+09
4882,5333036,awesome lol,0,1.760623e+09
51418,5540638,racial profile n nyou mean like epd arrest two...,1,1.760623e+09


**Feature store and Feature groups**  
The preprocessed features are organized into separate feature groups corresponding to train, validation, and test, and stored in feature stores. The Feature Store provides a centralized repository of ready-to-use features that can support varying modeling requirements. For example, in addition to the feature groups created for binary classification (toxic vs. non-toxic), a separate feature group can be designed to store features tailored for multi-class toxicity classification.

In [26]:
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)
   
feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

# Define Feature groups
train_feature_group_name = "train-feature-group"
validation_feature_group_name = "validation-feature-group"
test_feature_group_name = "test-feature-group"

# initialize feature group
train_feature_group = FeatureGroup(name=train_feature_group_name, sagemaker_session=feature_store_session)
validation_feature_group = FeatureGroup(name=validation_feature_group_name, sagemaker_session=feature_store_session)
test_feature_group = FeatureGroup(name=test_feature_group_name, sagemaker_session=feature_store_session)

# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
train_feature_group.load_feature_definitions(data_frame=train_df)
validation_feature_group.load_feature_definitions(data_frame=val_df)
test_feature_group.load_feature_definitions(data_frame=test_df)

# Check if the feature group exists
train_feature_group_exists = featureGroupExists(train_feature_group_name)
validation_feature_group_exists = featureGroupExists(validation_feature_group_name)
test_feature_group_exists = featureGroupExists(test_feature_group_name)

# Create feature groups
create_feature_group(train_feature_group,train_feature_group_exists)
create_feature_group(validation_feature_group, validation_feature_group_exists)
create_feature_group(test_feature_group,test_feature_group_exists) 


train-feature-group
✅ Feature group exists. Status: Created
validation-feature-group
✅ Feature group exists. Status: Created
test-feature-group
✅ Feature group exists. Status: Created
train-feature-group
✅ Feature group exists. Status: Created
train-feature-group already exists
validation-feature-group
✅ Feature group exists. Status: Created
validation-feature-group already exists
test-feature-group
✅ Feature group exists. Status: Created
test-feature-group already exists


In [27]:
#train_feature_group.ingest(data_frame=train_df, max_workers=3, wait=True) 

In [28]:
#validation_feature_group.ingest(data_frame=val_df, max_workers=3, wait=True) 

In [29]:
#test_feature_group.ingest(data_frame=test_df, max_workers=3, wait=True)

In [30]:
# Upload to S3
import s3fs

s3 = boto3.client('s3')

train_s3_resolved_uri = (
    train_feature_group.describe()
        .get("OfflineStoreConfig")
        .get('S3StorageConfig')
        .get('ResolvedOutputS3Uri')
)

train_feature_data = pd.read_parquet(train_s3_resolved_uri)
print(train_feature_data.shape)
train_feature_data.to_parquet("civil/train_feature.parquet")


(50540, 11)


In [31]:
val_s3_resolved_uri = (
    validation_feature_group.describe()
        .get("OfflineStoreConfig")
        .get('S3StorageConfig')
        .get('ResolvedOutputS3Uri')
)

validation_feature_data = pd.read_parquet(val_s3_resolved_uri)
print(validation_feature_data.shape)
validation_feature_data.to_parquet("civil/val_feature.parquet")


(12635, 11)


In [32]:
test_s3_resolved_uri = (
    test_feature_group.describe()
        .get("OfflineStoreConfig")
        .get('S3StorageConfig')
        .get('ResolvedOutputS3Uri')
)

test_feature_data = pd.read_parquet(test_s3_resolved_uri)
print(test_feature_data.shape)
test_feature_data.to_parquet("civil/test_feature.parquet")


(3325, 11)


In [33]:
train_feature_uri = sagemaker.s3.S3Uploader.upload(local_path='civil/train_feature.parquet', desired_s3_uri=base_uri)
val_feature_uri = sagemaker.s3.S3Uploader.upload(local_path='civil/val_feature.parquet', desired_s3_uri=base_uri)
test_feature_uri = sagemaker.s3.S3Uploader.upload(local_path='civil/test_feature.parquet', desired_s3_uri=base_uri)

print(train_feature_uri)
print(val_feature_uri)
print(test_feature_uri)

s3://sagemaker-us-east-1-046536196377/toxicity-CICD/train_feature.parquet
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/val_feature.parquet
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/test_feature.parquet


## Set up CI/CD pipeline

The Sagmeaker CI/CD pipeline automates the key components of the machine learning workflow. Specifically, the pipeline orchestrates the following stages:  
  
1. Data Transformation
2. Model Training
3. Model Evalutaion
4. Model Registry

#### Define Parameters to Parametrize Pipeline Execution

In [34]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputData",
    default_value=input_data_uri,
)
batch_data = ParameterString(
    name="BatchData",
    default_value=batch_data_uri,
)
accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.60)

In [35]:
from sagemaker.sklearn.processing import SKLearnProcessor

framework_version = "1.2-1"

sklearn_processor = SKLearnProcessor(
    framework_version=framework_version,
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="sklearn-toxicity-process",
    role=role,
    sagemaker_session=pipeline_session,
)

### Feature Engieneering/Data Transformation

The preprocessed data is fetched from the corresponding feature store and is vectorized using SKlearn’s TFIDF Vectorizer. The vectorization is subsequently applied to validation and test datasets. The vectorized data is stored in S3 bucket, which can serve as a checkpoint for further steps in the pipeline.


In [36]:
%%writefile code/preprocessing.py

import subprocess
import sys


print("starting preprocessing...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "s3fs"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "sagemaker"])

import s3fs
import numpy as np
import pandas as pd
import pickle
import sagemaker
from sagemaker.session import Session
from sagemaker.s3 import S3Uploader
import os, boto3
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer


os.environ["AWS_DEFAULT_REGION"] = "us-east-1"   
os.environ["AWS_REGION"] = os.environ["AWS_DEFAULT_REGION"]

# now create sessions/clients
boto_sess = boto3.Session(region_name=os.environ["AWS_DEFAULT_REGION"])
sm_sess = Session(boto_sess)


print("completed installation of libraries")


train_s3_resolved_uri = "s3://sagemaker-us-east-1-046536196377/toxicity-CICD/train_feature.parquet"
val_s3_resolved_uri = "s3://sagemaker-us-east-1-046536196377/toxicity-CICD/val_feature.parquet"
test_s3_resolved_uri = "s3://sagemaker-us-east-1-046536196377/toxicity-CICD/test_feature.parquet"

train_feature_data = pd.read_parquet(train_s3_resolved_uri)
validation_feature_data = pd.read_parquet(val_s3_resolved_uri)
test_feature_data = pd.read_parquet(test_s3_resolved_uri)

X_train, y_train = train_feature_data["comment_text"], train_feature_data["toxicity"]
X_val, y_val = validation_feature_data["comment_text"], validation_feature_data["toxicity"]
X_test, y_test = test_feature_data["comment_text"], test_feature_data["toxicity"]

vectorizer = TfidfVectorizer(max_features=1000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("completed vectorizing text")

# Combine features and labels
train_combined = pd.DataFrame(np.hstack([y_train.values.reshape(-1, 1), X_train_tfidf.toarray()]))
val_combined = pd.DataFrame(np.hstack([y_val.values.reshape(-1, 1), X_val_tfidf.toarray()]))
test_combined = pd.DataFrame(np.hstack([y_test.values.reshape(-1, 1), X_test_tfidf.toarray()]))

# write to csv
train_combined.to_csv('/opt/ml/processing/train/train.csv', index=False, header=False)
val_combined.to_csv('/opt/ml/processing/train/validation.csv', index=False, header=False)
test_combined.to_csv('/opt/ml/processing/train/test.csv', index=False, header=False)

# 'vectorizer' is used to vectorize train dataset
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

s3_prefix = "s3://sagemaker-us-east-1-046536196377/toxicity-CICD"  # note: no filename here

out_uri = S3Uploader.upload(local_path="vectorizer.pkl",
                            desired_s3_uri=s3_prefix,
                            sagemaker_session=sm_sess)
print(out_uri)


print("completed preprocessing step")


Overwriting code/preprocessing.py


In [37]:
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep


processor_args = sklearn_processor.run(
    code="code/preprocessing.py",
    # No ProcessingInput needed!
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ]
)

step_process = ProcessingStep(name="toxicityprocess", step_args=processor_args)

### Model Training

The model leverages SageMaker’s XGBoost algorithm to train the toxicity data. The training was configured with the hyperparameters to optimize learning outcomes.


In [38]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.functions import Join

model_path = base_uri
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.xlarge",
)

xgb_train = Estimator(
    image_uri=image_uri,
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    role=role,
    sagemaker_session=pipeline_session,
)

xgb_train.set_hyperparameters(
    objective="binary:logistic",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    num_round=100,
    eval_metric= "auc"
)

# This is the single S3 prefix created by the Processing output named "train"
train_output_prefix = step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
#validation_output_prefix = step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri

train_args = xgb_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=Join(on='/', values=[train_output_prefix, "train.csv"]),
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=Join(on='/', values=[train_output_prefix, "validation.csv"]),
            content_type="text/csv",
        ),
    }
)


In [39]:
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep


step_train = TrainingStep(
    name="ToxicityTrain",
    step_args=train_args,
)

### Model Evaluation

The inference is performed using the test dataset, and the model is evaluated based on its accuracy, precision, recall, and F1 score. The results of the evaluation job are stored in evaluation.json.


#### Define a Model Evaluation Step to Evaluate the Trained Model

In [40]:
%%writefile code/evaluation.py
import json
import pathlib
import pickle
import tarfile

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if __name__ == "__main__":
    model_path = f"/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")

    #model = pickle.load(open("xgboost-model", "rb"))
    model = xgb.Booster()
    model.load_model("xgboost-model")


    test_path = "/opt/ml/processing/test/test.csv"
    df = pd.read_csv(test_path, header=None)

    y_test = df.iloc[:, 0].to_numpy()
    df.drop(df.columns[0], axis=1, inplace=True)

    X_test = xgb.DMatrix(df.values)

    raw_predictions = model.predict(X_test)

    # Determine if this is binary or multiclass
    if len(np.unique(y_test)) > 2:
        # Multiclass classification
        y_pred = np.argmax(raw_predictions, axis=1) if raw_predictions.ndim > 1 else raw_predictions.astype(int)
        average_method = "macro"
    else:
        # Binary classification; with XGBoost/SageMaker, output may be probabilities
        y_pred = (raw_predictions > 0.5).astype(int)
        average_method = "binary"

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average=average_method, zero_division=0)
    recall = recall_score(y_test, y_pred, average=average_method, zero_division=0)
    f1 = f1_score(y_test, y_pred, average=average_method, zero_division=0)

    report_dict = {
        "classification_metrics": {
            "accuracy": {"value": accuracy},
            "precision": {"value": precision},
            "recall": {"value": recall},
            "f1": {"value": f1},
        },
    }

    output_dir = "/opt/ml/processing/evaluation"
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    evaluation_path = f"{output_dir}/evaluation.json"
    with open(evaluation_path, "w") as f:
        f.write(json.dumps(report_dict))


Overwriting code/evaluation.py


In [41]:
from sagemaker.processing import ScriptProcessor


script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-toxicity-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

test_output_prefix = step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri


eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=Join(on='/', values=[test_output_prefix, "test.csv"]),
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/evaluation.py",
)

In [42]:
from sagemaker.workflow.properties import PropertyFile


evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)
step_eval = ProcessingStep(
    name="ToxicityEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

### Model Registry

#### Define a Create Model Step to Create a Model

In [43]:
from sagemaker.model import Model

model = Model(
    image_uri=image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
)

In [44]:
from sagemaker.inputs import CreateModelInput
from sagemaker.workflow.model_step import ModelStep

step_create_model = ModelStep(
    name="ToxicityCreateModel",
    step_args=model.create(instance_type="ml.m5.large", accelerator_type="ml.eia1.medium"),
)

#### Define a Register Model Step to Create a Model Package¶

In [45]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="ToxicityRegisterModel", step_args=register_args)

The ToxicityAccuracyCond job, illustrated in Figure 1.1, is responsible for evaluating the model’s accuracy against a predefined threshold. If the measured accuracy falls below 0.65, the condition job triggers a failure, thereby aborting the pipeline. If the threshold is above 0.65, the pipeline successfully registers the model in the model registry. 


#### Define a Fail Step to Terminate the Pipeline Execution and Mark it as Failed

In [46]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

step_fail = FailStep(
    name="ToxicityFail",
    error_message=Join(on=" ", values=["Execution failed due to Accuracy <", accuracy_threshold]),
)

#### Define a Condition Step to Check Accuracy and Conditionally Create a Model and Run a Batch Transformation and Register a Model in the Model Registry, Or Terminate the Execution in Failed State

In [47]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet


cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.accuracy.value",
    ),
    right=accuracy_threshold,
)

step_cond = ConditionStep(
    name="ToxicityAccuracyCond",
    conditions=[cond_gte],
    if_steps=[step_register, step_create_model],
    else_steps=[step_fail],
)

### Initiate Pipeline Execution

#### Define a Pipeline of Parameters, Steps, and Conditions

In [48]:
from sagemaker.workflow.pipeline import Pipeline


pipeline_name = f"ToxicityPipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        batch_data,
        accuracy_threshold,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
)

#### Submit the pipeline to SageMaker and start execution

In [49]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:046536196377:pipeline/ToxicityPipeline',
 'ResponseMetadata': {'RequestId': '89f1da8d-bf63-4f2d-a20a-f23b640fde2d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '89f1da8d-bf63-4f2d-a20a-f23b640fde2d',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '107',
   'date': 'Thu, 16 Oct 2025 14:02:10 GMT'},
  'RetryAttempts': 0}}

In [50]:
execution = pipeline.start()

In [51]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:046536196377:pipeline/ToxicityPipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:046536196377:pipeline/ToxicityPipeline/execution/2ku2eneovgu4',
 'PipelineExecutionDisplayName': 'execution-1760623330345',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2025, 10, 16, 14, 2, 10, 248000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2025, 10, 16, 14, 2, 10, 248000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:046536196377:user-profile/d-rkebokcet1uv/karthik',
  'UserProfileName': 'karthik',
  'DomainId': 'd-rkebokcet1uv',
  'IamIdentity': {'Arn': 'arn:aws:sts::046536196377:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROAQVVOGAEM5ZMZ37A2C:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:046536196377:user-profile/d-rkebokcet1uv/karthik',
  'UserProfileName': 'karthik',
  'DomainId': 'd-rkebokcet1uv',
  'IamIdentity': {

In [52]:
execution.wait()

In [53]:
execution.list_steps()

[{'StepName': 'ToxicityRegisterModel-RegisterModel',
  'StartTime': datetime.datetime(2025, 10, 16, 14, 13, 42, 763000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 44, 417000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'RegisterModel': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:model-package/ToxicityModelPackageGroup/4'}},
  'AttemptCount': 1},
 {'StepName': 'ToxicityCreateModel-CreateModel',
  'StartTime': datetime.datetime(2025, 10, 16, 14, 13, 42, 763000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 44, 697000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Model': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:model/pipelines-2ku2eneovgu4-ToxicityCreateModel--7WZ9JTWy1W'}},
  'AttemptCount': 1},
 {'StepName': 'ToxicityAccuracyCond',
  'StartTime': datetime.datetime(2025, 10, 16, 14, 13, 41, 472000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 41, 672000, 

### Examining the Evaluation

In [54]:
from pprint import pprint


evaluation_json = sagemaker.s3.S3Downloader.read_file(
    "{}/evaluation.json".format(
        step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
    )
)
pprint(json.loads(evaluation_json))

{'classification_metrics': {'accuracy': {'value': 0.7500751879699248},
                            'f1': {'value': 0.7256520303730605},
                            'precision': {'value': 0.8039502560351134},
                            'recall': {'value': 0.661251504211793}}}


### Lineage


In [55]:
import time
from sagemaker.lineage.visualizer import LineageTableVisualizer


viz = LineageTableVisualizer(sagemaker.session.Session())
for execution_step in reversed(execution.list_steps()):
    print(execution_step)
    display(viz.show(pipeline_execution_step=execution_step))
    time.sleep(5)

{'StepName': 'toxicityprocess', 'StartTime': datetime.datetime(2025, 10, 16, 14, 2, 11, 221000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2025, 10, 16, 14, 7, 16, 936000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:processing-job/pipelines-2ku2eneovgu4-toxicityprocess-qdPIZb4ftU'}}, 'AttemptCount': 1}


,Name/Source,Direction,Type,Association Type,Lineage Type
0,s3://...afb5583ac6812b141bbc6ae/preprocessing.py,Input,DataSet,ContributedTo,artifact
1,68331...com/sagemaker-scikit-learn:1.2-1-cpu-py3,Input,Image,ContributedTo,artifact
2,s3://...2ku2eneovgu4/toxicityprocess/output/test,Output,DataSet,Produced,artifact
3,s3://...eovgu4/toxicityprocess/output/validation,Output,DataSet,Produced,artifact
4,s3://...ku2eneovgu4/toxicityprocess/output/train,Output,DataSet,Produced,artifact


{'StepName': 'ToxicityTrain', 'StartTime': datetime.datetime(2025, 10, 16, 14, 7, 17, 462000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2025, 10, 16, 14, 11, 6, 604000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:training-job/pipelines-2ku2eneovgu4-ToxicityTrain-csHrwAoZnf'}}, 'AttemptCount': 1}


,Name/Source,Direction,Type,Association Type,Lineage Type
0,s3://...icityprocess/output/train/validation.csv,Input,DataSet,ContributedTo,artifact
1,s3://...4/toxicityprocess/output/train/train.csv,Input,DataSet,ContributedTo,artifact
2,68331...-1.amazonaws.com/sagemaker-xgboost:1.7-1,Input,Image,ContributedTo,artifact
3,s3://...cityTrain-csHrwAoZnf/output/model.tar.gz,Output,Model,Produced,artifact


{'StepName': 'ToxicityEval', 'StartTime': datetime.datetime(2025, 10, 16, 14, 11, 7, 441000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 40, 767000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:processing-job/pipelines-2ku2eneovgu4-ToxicityEval-qWcosBiGtz'}}, 'AttemptCount': 1}


...

None

{'StepName': 'ToxicityCreateModel-CreateModel', 'StartTime': datetime.datetime(2025, 10, 16, 14, 13, 42, 763000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 44, 697000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'Model': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:model/pipelines-2ku2eneovgu4-ToxicityCreateModel--7WZ9JTWy1W'}}, 'AttemptCount': 1}


None

{'StepName': 'ToxicityRegisterModel-RegisterModel', 'StartTime': datetime.datetime(2025, 10, 16, 14, 13, 42, 763000, tzinfo=tzlocal()), 'EndTime': datetime.datetime(2025, 10, 16, 14, 13, 44, 417000, tzinfo=tzlocal()), 'StepStatus': 'Succeeded', 'Metadata': {'RegisterModel': {'Arn': 'arn:aws:sagemaker:us-east-1:046536196377:model-package/ToxicityModelPackageGroup/4'}}, 'AttemptCount': 1}


,Name/Source,Direction,Type,Association Type,Lineage Type
0,s3://...cityTrain-csHrwAoZnf/output/model.tar.gz,Input,Model,ContributedTo,artifact
1,68331...-1.amazonaws.com/sagemaker-xgboost:1.7-1,Input,Image,ContributedTo,artifact
2,ToxicityModelPackageGroup-4-1760624024-aws-mod...,Input,ModelLifeCycle,ContributedTo,action
3,ToxicityModelPackageGroup-4-PendingManualAppro...,Input,Approval,ContributedTo,action
4,ToxicityModelPackageGroup-1760505307-aws-model...,Output,ModelGroup,AssociatedWith,context


## Model Deployment

The trained model is retrieved from the model registry and deployed to an AWS SageMaker endpoint. This endpoint exposes a RESTful API that external HTTP-based applications, such as online news platforms, can consume.

In [12]:
from datetime import datetime

# S3 prefixes
data_capture_prefix = f"{base_uri}/datacapture"
s3_capture_upload_path = f"{data_capture_prefix}"

ground_truth_upload_path = (
    f"{base_uri}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
)

reports_prefix = "reports"
s3_report_path = f"{base_uri}/{reports_prefix}"

##Get the model monitor image
monitor_image_uri = sagemaker.image_uris.retrieve(framework="model-monitor", region=region)

print("Image URI:", monitor_image_uri)
print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

Image URI: 156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer
Capture path: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/datacapture
Ground truth path: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/ground_truth_data/2025-10-18-21-08-59
Report path: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/reports


In [13]:
from sagemaker.estimator import Estimator

job_name = "pipelines-2s8g33l3enzf-ToxicityTrain-8k3t0Q9XZJ"
output_location = "{}/output/{}".format(base_uri, job_name)
print(output_location)

xgb_model = Estimator.attach(job_name, sagemaker_session=sagemaker_session)
print(xgb_model.model_data)  # same S3 path


s3://sagemaker-us-east-1-046536196377/toxicity-CICD/output/pipelines-2s8g33l3enzf-ToxicityTrain-8k3t0Q9XZJ

2025-10-16 00:32:43 Starting - Preparing the instances for training
2025-10-16 00:32:43 Downloading - Downloading the training image
2025-10-16 00:32:43 Training - Training image download completed. Training in progress.
2025-10-16 00:32:43 Uploading - Uploading generated training model
2025-10-16 00:32:43 Completed - Training job completed
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/pipelines-2s8g33l3enzf-ToxicityTrain-8k3t0Q9XZJ/output/model.tar.gz


In [14]:
from sagemaker.model_monitor import DataCaptureConfig
import datetime
from datetime import datetime, timezone


# Deploy Model
endpoint_name = f"Toxicity-xgb-churn-model-quality-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)

data_capture_config = DataCaptureConfig(
    enable_capture=True, sampling_percentage=100, destination_s3_uri=s3_capture_upload_path
)

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
)

/tmp/ipykernel_151/2722074575.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  endpoint_name = f"Toxicity-xgb-churn-model-quality-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"


EndpointName = Toxicity-xgb-churn-model-quality-monitor-2025-10-18-2109
-----!

In [110]:
#abs Create SageMaker client
sagemaker_client = boto3.client('sagemaker', region_name=region)

response = sagemaker_client.list_endpoints()

for ep in response["Endpoints"]:
    print(f"Endpoint Name: {ep['EndpointName']}, Status: {ep['EndpointStatus']}")

endpoint_name = ep['EndpointName']

Endpoint Name: Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745, Status: InService


### Create the SageMaker Predictor object from the endpoint to be used for invoking the model¶

In [111]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

In [112]:
from sagemaker.predictor import Predictor

predictor = Predictor(
    endpoint_name=endpoint_name, sagemaker_session=sagemaker_session, serializer=CSVSerializer()
)

In [113]:
test_input = pd.read_csv('civil/batch_data.csv')
test_input=test_input[['comment_text', 'toxicity']]
test_input = test_input[:200]
print(test_input.shape)
test_input.to_csv("civil/test_input.csv", index=False, header=False)
test_input.head()

(200, 2)


,comment_text,toxicity
0,Why would N Korea fire a nuclear missile again...,0
1,Canada is doomed under this brainless muslim!,1
2,Defense spending is enumerated responsibility ...,1
3,Being right about what? \n\nBill is in trouble...,1
4,A vote for Hillary is a vote for globalism and...,0


In [114]:
test_input[test_input['toxicity']=='toxicity']

,comment_text,toxicity


In [115]:
from sagemaker.s3 import S3Uploader
# Upload to S3
s3 = boto3.client('s3')
test_vect_uri = S3Uploader.upload(local_path='civil/test_input.csv', desired_s3_uri=f"{base_uri}/test")
test_vect_uri

's3://sagemaker-us-east-1-046536196377/toxicity-CICD/test/test_input.csv'

In [116]:
import csv

with open("civil/test_input.csv", newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        input_cols = row[-2].strip()
        label = row[-1].strip()
        print(f"{input_cols}~{label}/n")
        if i == 4:
            break

Why would N Korea fire a nuclear missile against the US,  Japan or S Korea if they know that within minutes nuclear missiles would be raining down on their country in retaliation?  Worse yet if their missiles fall in the ocean way short to their targets.  Too bad, no second chance.  Hopefully Trump doesn,t start anything and open up a can of worms.  I don't think fat Kim will do anything huge, maybe small skirmishes.~0/n
Canada is doomed under this brainless muslim!~1/n
Defense spending is enumerated responsibility of the federal government, running our lives, or even our health care is not.
If you believe that the federal government should control our health care, work to amend the constitution.
 Don't ignore it, or rely on some idiot in a black robe to basterdize it.
23 days !!!~1/n
Being right about what? 

Bill is in trouble for sexually harassing women. He just settled a lawsuit about other sexual harassment. 

His political slant is not he issue here. You are conflating two very 

In [117]:
import csv
import pickle

toxicity_cutoff = 0.70

def preprocess(text_list):
    tfidf_features = vectorizer.transform(text_list)
    # Convert to dense array and CSV string format (no headers)
    csv_features = ",".join(map(str, tfidf_features.toarray().flatten()))
    return csv_features

s3_bucket = "sagemaker-us-east-1-046536196377"
s3_key = "toxicity-CICD/vectorizer.pkl"
local_path = "civil/vectorizer.pkl"
boto3.client('s3').download_file(s3_bucket, s3_key, local_path)

# Load after downloading
with open(local_path, 'rb') as f:
    vectorizer = pickle.load(f)
    
limit = 200  # Need at least 200 samples to compute standard deviations
i = 0
with open("civil/test_predictions.csv", "w") as baseline_file:
    baseline_file.write("probability,prediction,label\n")  # our header
    with open("civil/test_input.csv", "r") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            input_cols = row[-2].strip()
            label = row[-1].strip()
            #print(f"{input_cols}~{label}/n")
            input_texts = [input_cols]
            payload = "\n".join([preprocess([text]) for text in input_texts])
            probability = float(predictor.predict(payload))
            prediction = "1" if probability > toxicity_cutoff else "0"
            baseline_file.write(f"{probability},{prediction},{label}\n")
            #print(f"{probability},{prediction},{label}\n")
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            time.sleep(0.5)
print()
print("Done!")

........................................................................................................................................................................................................
Done!


## Model Monitoring

The project employs Amazon SageMaker Model Monitor to maintain the performance of the Toxicity classification model. The monitoring job establishes a baseline model performance using ground truth labels and continuously evaluates subsequent inferences made through the deployed endpoint. This process enables the detection of model drift, ensuring that any deviations in input data or prediction quality are promptly identified. By tracking performance characteristics such as accuracy, precision, and other key metrics in real time, Model Monitor provides an effective mechanism to sustain model quality. 

### Generate a baseline for model quality performance 

In [118]:
from sagemaker.s3 import S3Downloader, S3Uploader

baseline_prefix = "baselining"
baseline_data_prefix = "data"
baseline_results_prefix = "results"

baseline_data_uri = f"{base_uri}/{baseline_data_prefix}"
baseline_results_uri = f"{base_uri}/{baseline_results_prefix}"
print(f"Baseline data uri: {baseline_data_uri}")
print(f"Baseline results uri: {baseline_results_uri}")

Baseline data uri: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/data
Baseline results uri: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/results


In [119]:
baseline_dataset_uri = S3Uploader.upload(f"civil/test_predictions.csv", baseline_data_uri)
baseline_dataset_uri

's3://sagemaker-us-east-1-046536196377/toxicity-CICD/data/test_predictions.csv'

### Create a baselining job with validation dataset predictions

In [120]:
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor import EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

In [121]:
# Create the model quality monitoring object
churn_model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [122]:
from datetime import datetime, timezone

# Name of the model quality baseline job
baseline_job_name = f"Toxicity-xgb-churn-model-baseline-job-{datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S")}"
baseline_job_name

'Toxicity-xgb-churn-model-baseline-job-2025-10-16-17-50-31'

In [123]:
baseline_dataset_uri

's3://sagemaker-us-east-1-046536196377/toxicity-CICD/data/test_predictions.csv'

In [124]:
predicitons = pd.read_csv("civil/test_predictions.csv")
predicitons.head()

,probability,prediction,label
0,0.399400,0,0
1,0.597691,0,1
2,0.969142,1,1
3,0.297202,0,1
4,0.368588,0,0


In [125]:
# Execute the baseline suggestion job.
job = churn_model_quality_monitor.suggest_baseline(
    job_name=baseline_job_name,
    baseline_dataset=baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
)
job.wait(logs=False)

INFO:sagemaker:Creating processing-job with name Toxicity-xgb-churn-model-baseline-job-2025-10-16-17-50-31


...........................................................!

### Explore the results of the baselining job

### View the metrics generated

In [126]:
baseline_job = churn_model_quality_monitor.latest_baselining_job
baseline_job

In [127]:
binary_metrics = baseline_job.baseline_statistics().body_dict["binary_classification_metrics"]
pd.json_normalize(binary_metrics).T

,0
confusion_matrix.0.0,100
confusion_matrix.0.1,11
confusion_matrix.1.0,52
confusion_matrix.1.1,37
recall.value,0.41573
recall.standard_deviation,0.024534
precision.value,0.770833
precision.standard_deviation,0.008123
accuracy.value,0.685
accuracy.standard_deviation,0.024515


In [128]:
pd.DataFrame(baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

,threshold,comparison_operator
recall,0.41573,LessThanThreshold
precision,0.770833,LessThanThreshold
accuracy,0.685,LessThanThreshold
true_positive_rate,0.41573,LessThanThreshold
true_negative_rate,0.900901,LessThanThreshold
false_positive_rate,0.099099,GreaterThanThreshold
false_negative_rate,0.58427,GreaterThanThreshold
auc,0.747697,LessThanThreshold
f0_5,0.658363,LessThanThreshold
f1,0.540146,LessThanThreshold


In [129]:
# prepare test data to generate predictions
test_input = pd.read_csv(f'{base_uri}/batch_data.csv')
test_input=test_input[['comment_text', 'toxicity']]
test_input_2 = test_input[200:500]
print(test_input_2.shape)
test_input_2.to_csv('civil/test_input_2.csv', index=False, header=False)
test_input_2.head()

(300, 2)


,comment_text,toxicity
200,She already looks like she's in her 60s,0
201,Tyipical leftest and their “identity liberalis...,0
202,A majority of Students aged 12-18 are smoking ...,0
203,To Tory: First you don't know for a fact wheth...,0
204,AR's account for a fraction of less than 1 per...,0


### Real Time Inference

To simulate real-time inference, a dedicated thread is initiated that interacts with the deployed SageMaker endpoint. The downstream applications invoke this model endpoint to generate predictions. The predictor object of the endpoint predicts the toxicity label for the incoming comment text. Based on this label, the application determines whether the comment should be flagged for deletion or allowed to remain.


In [130]:
threshold_limit=0.70

from threading import Thread
import csv
import time

sm_runtime = boto3.client('sagemaker-runtime')

def preprocess(text_list):
    tfidf_features = vectorizer.transform(text_list)
    # Convert to dense array and CSV string format (no headers)
    csv_features = ",".join(map(str, tfidf_features.toarray().flatten()))
    return csv_features

def get_payload(input_cols):
    input_texts = [input_cols]
    payload = "\n".join([preprocess([text]) for text in input_texts])
    return payload
    
def invoke_endpoint(ep_name, file_name, log_filename):
    with open(file_name, "r") as f, open(log_filename, "w", newline="") as outfile:
        writer = csv.writer(outfile)
        writer.writerow(["comment_text", "inference_id", "prediction", "probability"])
        i = 0
        for row in f:
            input_cols = row[-2].strip()
            label = row[-1].strip()
            payload = get_payload(input_cols) # convert to feature vector
            response = sagemaker_session.sagemaker_runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Body=payload,
                InferenceId=str(i),  # unique ID per row
            )["Body"].read().decode("utf-8")
            probability = float(response.strip())
            prediction = "1" if probability > threshold_limit else "0"
            # Log the result including original comment text
            writer.writerow([input_cols, i, prediction, probability])
            i += 1
            time.sleep(1)


def invoke_endpoint_forever():
    while True:
        try:
            invoke_endpoint(endpoint_name, "civil/test_input_2.csv", "civil/results_with_text.csv")
        except sagemaker_session.sagemaker_runtime_client.exceptions.ValidationError:
            pass


thread = Thread(target=invoke_endpoint_forever)
thread.start()


Exception in thread Thread-6 (invoke_endpoint_forever):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/opt/conda/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_158/958470068.py", line 46, in invoke_endpoint_forever
  File "/tmp/ipykernel_158/958470068.py", line 26, in invoke_endpoint
IndexError: string index out of range


### View captured data

In [131]:
print("Waiting for captures to show up", end="")
for _ in range(120):
    capture_files = sorted(S3Downloader.list(f"{s3_capture_upload_path}/{endpoint_name}"))
    if capture_files:
        capture_file = S3Downloader.read_file(capture_files[-1]).split("\n")
        capture_record = json.loads(capture_file[0])
        if "inferenceId" in capture_record["eventMetadata"]:
            break
    print(".", end="", flush=True)
    time.sleep(1)
print()
print("Found Capture Files:")
print("\n ".join(capture_files[-3:]))

Waiting for captures to show up.............................................................
Found Capture Files:
s3://sagemaker-us-east-1-046536196377/toxicity-CICD/datacapture/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/AllTraffic/2025/10/16/17/48-47-760-cfba086b-2622-435a-8058-a0fe68ce76c1.jsonl
 s3://sagemaker-us-east-1-046536196377/toxicity-CICD/datacapture/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/AllTraffic/2025/10/16/17/49-47-826-11dc4a7e-1699-4cf3-9a47-9bcdc1c495ab.jsonl
 s3://sagemaker-us-east-1-046536196377/toxicity-CICD/datacapture/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/AllTraffic/2025/10/16/17/55-34-606-7c024d24-6b6b-43c9-93d9-930f861ecb65.jsonl


In [132]:
print("\n".join(capture_file[-3:-1]))

{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [133]:
print(json.dumps(capture_record, indent=2))

{
  "captureData": {
    "endpointInput": {
      "observedContentType": "text/csv",
      "mode": "INPUT",
      "data": "0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

### Build Ground Truth Records from CSV

In [134]:
csv_path = "civil/test_input_2.csv"

def ground_truth_with_label(inference_id, true_label):
    return {
        "groundTruthData": {
            "data": str(true_label),  # Use label from your CSV
            "encoding": "CSV",
        },
        "eventMetadata": {
            "eventId": str(inference_id),
        },
        "eventVersion": "0",
    }

def generate_gt_records_from_csv(csv_path):
    gt_records = []
    with open(csv_path, "r") as f:
        reader = csv.reader(f)
        for idx, row in enumerate(reader):
            if len(row) >= 2:
                input_cols = row[-2].strip()
                label = row[-1].strip()
                gt_record = ground_truth_with_label(idx, label)
                gt_records.append(gt_record)
    return gt_records


In [135]:
def upload_ground_truth(records, upload_time):
    fake_records = [json.dumps(r) for r in records]
    data_to_upload = "\n".join(fake_records)
    target_s3_uri = f"{ground_truth_upload_path}/{upload_time:%Y/%m/%d/%H/%M%S}.jsonl"
    print(f"Uploading {len(fake_records)} records to", target_s3_uri)
    S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)

# Example usage:
gt_records = generate_gt_records_from_csv(csv_path)
upload_ground_truth(gt_records, datetime.utcnow())


Uploading 300 records to s3://sagemaker-us-east-1-046536196377/toxicity-CICD/ground_truth_data/2025-10-16-17-45-08/2025/10/16/17/5652.jsonl


/tmp/ipykernel_158/898246835.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  upload_ground_truth(gt_records, datetime.utcnow())


In [136]:
gt_records

[{'groundTruthData': {'data': '0', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '0'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '0', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '1'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '0', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '2'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '0', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '3'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '0', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '4'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '1', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '5'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '1', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '6'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '1', 'encoding': 'CSV'},
  'eventMetadata': {'eventId': '7'},
  'eventVersion': '0'},
 {'groundTruthData': {'data': '1', 'encoding': 'CSV'},
  'eventMetadata'

### Create a monitoring schedule

A monitoring schedule is established to monitor the model performance on an hourly basis. The accuracy threshold limit for the baseline performance is set at 65%. Ground truth labels are created for the incoming payload. Whenever the model makes an inference, if the monitor detects accuracy degradation below 65%, the monitor raises a CloudWatch alarm notifying a drift in model performance.



In [137]:
# Monitoring schedule name
churn_monitor_schedule_name = (
    f"Toxicity-xgb-churn-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"
)
churn_monitor_schedule_name

/tmp/ipykernel_158/2338135700.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"Toxicity-xgb-churn-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"


'Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756'

In [138]:
# Create an enpointInput
endpointInput = EndpointInput(
    endpoint_name=predictor.endpoint_name,
    probability_attribute="0",
    probability_threshold_attribute=0.70,
    destination="/opt/ml/processing/input_data",
)

In [139]:
churn_monitor_schedule_name

'Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756'

In [140]:
# Create the monitoring schedule to execute every hour.
from sagemaker.model_monitor import CronExpressionGenerator

response = churn_model_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=churn_monitor_schedule_name,
    endpoint_input=endpointInput,
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    ground_truth_input=ground_truth_upload_path,
    constraints=baseline_job.suggested_constraints(),
    schedule_cron_expression = CronExpressionGenerator.hourly(),
    #data_analysis_start_time="-PT1H",   # start = now minus 1 hour
    #data_analysis_end_time="PT0H",      # end   = now
    enable_cloudwatch_metrics=True,
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756


In [141]:
# Create the monitoring schedule
# You will see the monitoring schedule in the 'Scheduled' status
churn_model_quality_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:046536196377:monitoring-schedule/Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756',
 'MonitoringScheduleName': 'Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756',
 'MonitoringScheduleStatus': 'Pending',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2025, 10, 16, 17, 56, 53, 538000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2025, 10, 16, 17, 56, 53, 605000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2025-10-16-17-56-52-841',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745',
 'ResponseMetadata': {'RequestId': 'b30723be-08fa-479a-b67d-f3c31308b180',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b30723be-08fa-479a-b67d-f3c31308b180',
   'strict-transport-security': 'max-age=

### Examine monitoring schedule executions

In [142]:
# Initially there will be no executions since the first execution happens at the top of the hour
# Note that it is common for the execution to luanch upto 20 min after the hour.
executions = churn_model_quality_monitor.list_executions()
executions

[]

In [143]:
# Wait for the first execution of the monitoring_schedule
print("Waiting for first execution", end="")
while True:
    execution = churn_model_quality_monitor.describe_schedule().get(
        "LastMonitoringExecutionSummary"
    )
    if execution:
        break
    print(".", end="", flush=True)
    time.sleep(10)
print()
print("Execution found!")

Waiting for first execution..........................................................
Execution found!


In [144]:
while not executions:
    executions = churn_model_quality_monitor.list_executions()
    print(".", end="", flush=True)
    time.sleep(10)
latest_execution = executions[-1]
latest_execution.describe()

..................

{'ProcessingInputs': [{'InputName': 'groundtruth_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-046536196377/toxicity-CICD/ground_truth_data/2025-10-16-17-45-08/2025/10/16/17',
    'LocalPath': '/opt/ml/processing/groundtruth/2025/10/16/17',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}},
  {'InputName': 'endpoint_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-046536196377/toxicity-CICD/datacapture/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/AllTraffic/2025/10/16/17',
    'LocalPath': '/opt/ml/processing/input_data/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/AllTraffic/2025/10/16/17',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}}],
 'ProcessingOutputConfig': {'Outputs': [{'OutputName': 'result',
    'S3O

### Inspect a specific execution (latest execution)

In [145]:
status = execution["MonitoringExecutionStatus"]

while status in ["Pending", "InProgress"]:
    print("Waiting for execution to finish", end="")
    latest_execution.wait(logs=False)
    latest_job = latest_execution.describe()
    print()
    print(f"{latest_job['ProcessingJobName']} job status:", latest_job["ProcessingJobStatus"])
    print(
        f"{latest_job['ProcessingJobName']} job exit message, if any:",
        latest_job.get("ExitMessage"),
    )
    print(
        f"{latest_job['ProcessingJobName']} job failure reason, if any:",
        latest_job.get("FailureReason"),
    )
    time.sleep(
        30
    )  # model quality executions consist of two Processing jobs, wait for second job to start
    latest_execution = churn_model_quality_monitor.list_executions()[-1]
    execution = churn_model_quality_monitor.describe_schedule()["LastMonitoringExecutionSummary"]
    status = execution["MonitoringExecutionStatus"]

print("Execution status is:", status)

if status != "Completed":
    print(execution)
    print(
        "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
    )

Waiting for execution to finish........................................................!
groundtruth-merge-202510161800-83f00b08980f2ae22e2ca005 job status: Completed
groundtruth-merge-202510161800-83f00b08980f2ae22e2ca005 job exit message, if any: None
groundtruth-merge-202510161800-83f00b08980f2ae22e2ca005 job failure reason, if any: None
Waiting for execution to finish.....................................................!
model-quality-monitoring-202510161800-83f00b08980f2ae22e2ca005 job status: Completed
model-quality-monitoring-202510161800-83f00b08980f2ae22e2ca005 job exit message, if any: CompletedWithViolations: Job completed successfully with 8 violations.
model-quality-monitoring-202510161800-83f00b08980f2ae22e2ca005 job failure reason, if any: None
Execution status is: CompletedWithViolations
{'MonitoringScheduleName': 'Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756', 'ScheduledTime': datetime.datetime(2025, 10, 16, 18, 0, tzinfo=tzlocal()), 'CreationTime': datetime.

In [146]:
latest_execution = churn_model_quality_monitor.list_executions()[-1]
report_uri = latest_execution.describe()["ProcessingOutputConfig"]["Outputs"][0]["S3Output"][
    "S3Uri"
]
print("Report Uri:", report_uri)

Report Uri: s3://sagemaker-us-east-1-046536196377/toxicity-CICD/results/Toxicity-xgb-churn-model-quality-monitor-2025-10-16-1745/Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756/2025/10/16/18


### View violations generated by monitoring schedule

In [147]:
pd.options.display.max_colwidth = None
violations = latest_execution.constraint_violations().body_dict["violations"]
violations_df = pd.json_normalize(violations)
violations_df.head(10)

,constraint_check_type,description,metric_name
0,LessThanThreshold,Metric auc with 0.5 was LessThanThreshold '0.7476971353375845',auc
1,LessThanThreshold,Metric precision with 0.0 was LessThanThreshold '0.7708333333333334',precision
2,LessThanThreshold,Metric truePositiveRate with 0.0 was LessThanThreshold '0.4157303370786517',truePositiveRate
3,LessThanThreshold,Metric f1 with 0.0 was LessThanThreshold '0.5401459854014599',f1
4,LessThanThreshold,Metric accuracy with 0.46153846153846156 was LessThanThreshold '0.685',accuracy
5,GreaterThanThreshold,Metric falseNegativeRate with 1.0 was GreaterThanThreshold '0.5842696629213483',falseNegativeRate
6,LessThanThreshold,Metric recall with 0.0 was LessThanThreshold '0.4157303370786517',recall
7,LessThanThreshold,Metric f2 with 0.0 was LessThanThreshold '0.45792079207920794',f2


### Analyze model quality CloudWatch metrics 

#### List the CW metrics generated

In [148]:
# Create CloudWatch client
cw_client = boto3.Session().client("cloudwatch")

namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_dimensions = [
    {"Name": "Endpoint", "Value": endpoint_name},
    {"Name": "MonitoringSchedule", "Value": churn_monitor_schedule_name},
]

In [149]:
# List metrics through the pagination interface
paginator = cw_client.get_paginator("list_metrics")

for response in paginator.paginate(Dimensions=cw_dimensions, Namespace=namespace):
    model_quality_metrics = response["Metrics"]
    for metric in model_quality_metrics:
        print(metric["MetricName"])

f2_best_constant_classifier
f1_best_constant_classifier
recall_best_constant_classifier
au_prc
f0_5
f1
total_number_of_violations
true_negative_rate
f0_5_best_constant_classifier
f2
accuracy_best_constant_classifier
true_positive_rate
recall
auc
false_negative_rate
false_positive_rate
accuracy
precision_best_constant_classifier
precision


### Create a CloudWatch Alarm

In [150]:
alarm_name = "MODEL_QUALITY_F2_SCORE"
alarm_desc = (
    "Trigger an CloudWatch alarm when the f2 score drifts away from the baseline constraints"
)
mdoel_quality_f2_drift_threshold = (
    0.625  ##Setting this threshold purposefully low to see the alarm quickly.
)
metric_name = "f2"
namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    MetricName=metric_name,
    Namespace=namespace,
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": churn_monitor_schedule_name},
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=mdoel_quality_f2_drift_threshold,
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="breaching",
)

{'ResponseMetadata': {'RequestId': '8f84c705-6827-4e59-86eb-d185f6b80471',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '8f84c705-6827-4e59-86eb-d185f6b80471',
   'content-type': 'text/xml',
   'content-length': '214',
   'date': 'Thu, 16 Oct 2025 18:20:00 GMT'},
  'RetryAttempts': 0}}

In [ ]:
churn_model_quality_monitor.delete_monitoring_schedule()
time.sleep(60)  # actually wait for the deletion

INFO:sagemaker:Deleting Monitoring Schedule with name: Toxicity-xgb-churn-monitoring-schedule-2025-10-16-1756
INFO:sagemaker.model_monitor.model_monitoring:Deleting Model Quality Job Definition with name: model-quality-job-definition-2025-10-16-17-56-52-841
